1.How many tennis players are included in the dataset?

In [15]:
import pandas as pd
import pyarrow.parquet as pq
import glob
import os

base_path = '../data/input/tennis_data/'
all_files = glob.glob(os.path.join(base_path, '**/*.parquet'), recursive=True)

dfs = []

for i, file in enumerate(all_files[:10000]):
    try:
        schema = pq.read_schema(file)
        if 'player_id' in schema.names:
            df = pd.read_parquet(file, columns=['player_id'])
            dfs.append(df)
    except Exception as e:
        print(f"Skipped {file}: {e}")

combined_df = pd.concat(dfs, ignore_index=True)
combined_df.to_parquet('players_combined.parquet')


df = pd.read_parquet('players_combined.parquet')
num_players = df['player_id'].dropna().nunique()
print("Unique players:", num_players)


Unique players: 673


3.Which player has the highest number of wins? 

In [20]:
import glob
import os
import pandas as pd

base_path = '../data/input/tennis_data/'

# گرفتن همه فایل‌های پارکت توی پوشه‌های مختلف
all_files = glob.glob(os.path.join(base_path, '**/*.parquet'), recursive=True)

# بررسی 10 فایل اول برای پیدا کردن فایل حاوی نتایج مسابقات
for file in all_files[:10]:
    try:
        df = pd.read_parquet(file)
        print(f"File: {file}")
        print(df.columns)
        print(df.head(1))
        print('-'*40)
    except Exception as e:
        print(f"Error reading {file}: {e}")


File: ../data/input/tennis_data\20240201\data\raw\raw_match_parquet\away_team_11998445.parquet
Index(['match_id', 'name', 'slug', 'gender', 'user_count', 'residence',
       'birthplace', 'height', 'weight', 'plays', 'turned_pro',
       'current_prize', 'total_prize', 'player_id', 'current_rank',
       'name_code', 'country', 'full_name'],
      dtype='object')
   match_id                name                   slug gender  user_count  \
0  11998445  Auger-Aliassime F.  auger-aliassime-felix      M       23318   

             residence        birthplace  height  weight         plays  \
0  Monte Carlo, Monaco  Montreal, Canada    1.93      87  right-handed   

  turned_pro  current_prize  total_prize  player_id  current_rank name_code  \
0       2017         218538     10166964     192013            30       AUG   

  country               full_name  
0  Canada  Auger-Aliassime, Felix  
----------------------------------------
File: ../data/input/tennis_data\20240201\data\raw\raw_matc

In [21]:
import pyarrow.parquet as pq
import glob
from tqdm import tqdm
import pandas as pd

# همه فایل‌های مورد نظر (home و away)
away_files = glob.glob('../data/input/tennis_data/**/*away_team_*.parquet', recursive=True)
home_files = glob.glob('../data/input/tennis_data/**/*home_team_*.parquet', recursive=True)
file_paths = away_files + home_files

# لیست برای ذخیره نتایج
records = []

# با tqdm برای نوار پیشرفت
for path in tqdm(file_paths, desc="در حال خواندن فایل‌ها"):
    try:
        table = pq.read_table(path, columns=['player_id', 'full_name', 'total_prize'])
        df = table.to_pandas()
        if not df.empty and pd.notnull(df.iloc[0]['player_id']) and pd.notnull(df.iloc[0]['total_prize']):
            records.append({
                'player_id': df.iloc[0]['player_id'],
                'full_name': df.iloc[0]['full_name'],
                'total_prize': df.iloc[0]['total_prize']
            })
    except Exception as e:
        continue  # خطاهای احتمالی رو رد می‌کنیم

# تبدیل به DataFrame
result_df = pd.DataFrame(records)

# حذف رکوردهای خالی یا تکراری و پیدا کردن بیشترین total_prize
result_df.dropna(subset=['player_id', 'total_prize'], inplace=True)
result_df = result_df.sort_values('total_prize', ascending=False).drop_duplicates('player_id')

top_player = result_df.iloc[0]
print(f"\n🏆 بازیکنی با بیشترین total prize:\n{top_player['full_name']} — ${top_player['total_prize']:,}")


در حال خواندن فایل‌ها:  14%|█▎        | 16323/120030 [02:58<18:52, 91.57it/s] 


KeyboardInterrupt: 

5. How many sets are typically played in a tennis match? 

In [16]:
n = min(5, len(df))
sampled_df = df.sample(n)
display(sampled_df)


,player_id
714,230344
219,224033
1315,51651
180,369288
167,36300


In [23]:
from glob import glob

files = glob('../data/input/tennis_data/**/*.parquet')  # Update the path as needed

all_columns = set()

for file in files:
    df = pd.read_parquet(file)
    all_columns.update(df.columns)

print(all_columns)


set()


In [24]:
import pandas as pd
import os

folder_path = '../data/input/tennis_data/20240201/data/raw/raw_match_parquet'  # مسیر پوشه‌ت رو بزار اینجا
files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.parquet')]

target_cols = ['period_1', 'period_2', 'period_3', 'period_4', 'period_5']
files_with_all_periods = []

for file in files:
    try:
        df = pd.read_parquet(file, engine='pyarrow')
        if all(col in df.columns for col in target_cols):
            files_with_all_periods.append(file)
    except Exception as e:
        print(f'خطا در فایل {file}: {e}')

print(f'تعداد فایل‌هایی که همه ستون‌های period رو دارن: {len(files_with_all_periods)}')


تعداد فایل‌هایی که همه ستون‌های period رو دارن: 855


In [11]:
import pandas as pd

period_cols = ['period_1', 'period_2', 'period_3', 'period_4', 'period_5']
total_sets = []
total_matches = 0

for file in files_with_all_periods:
    df = pd.read_parquet(file, engine='pyarrow')
    
    # حذف ردیف‌هایی که همه periodهاش None هستن
    if df[period_cols].notna().sum(axis=1).sum() == 0:
        continue

    df['sets_played'] = df[period_cols].notna().sum(axis=1)
    total_sets.extend(df['sets_played'].tolist())
    total_matches += len(df)

# محاسبه‌ی میانگین ست‌ها
if total_sets:
    avg_sets = sum(total_sets) / len(total_sets)
    print(f'🎾 میانگین تعداد ست‌های بازی‌شده در مجموع {len(total_sets)} مسابقه: {avg_sets:.2f}')
else:
    print('هیچ مسابقه‌ای با اطلاعات ست معتبر پیدا نشد.')


NameError: name 'files_with_all_periods' is not defined

13. What is the distribution of left-handed versus right-handed players?  

In [19]:
import os
import pandas as pd
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

folder_path = r"C:\Users\ASUSE\Desktop\data-analysis-project\Data-Analysis-Mini-Project\data\input\tennis_data"

# گرفتن همه فایل‌های away_team
def list_away_team_files(path):
    return [
        os.path.join(root, file)
        for root, _, files in os.walk(path)
        for file in files
        if file.endswith(".parquet") and "away_team" in file
    ]

# پردازش هر فایل
def process_file(file_path):
    try:
        df = pd.read_parquet(file_path, columns=["plays"])
        return Counter(df["plays"].dropna())
    except:
        return Counter()

def main():
    files = list_away_team_files(folder_path)
    all_counts = Counter()

    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = {executor.submit(process_file, f): f for f in files}
        for f in tqdm(as_completed(futures), total=len(futures), desc="در حال پردازش سریع"):
            all_counts += f.result()

    total = sum(all_counts.values())
    percentages = {k: round((v / total) * 100, 2) for k, v in all_counts.items()}

    print("\n✅ تعداد بازیکنان به تفکیک دست بازی:")
    for k, v in all_counts.items():
        print(f"{k}: {v}")

    print("\n✅ درصد بازیکنان به تفکیک دست بازی:")
    for k, v in percentages.items():
        print(f"{k}: {v}%")

main()


در حال پردازش سریع: 0it [00:00, ?it/s]


✅ تعداد بازیکنان به تفکیک دست بازی:

✅ درصد بازیکنان به تفکیک دست بازی:
